# DSC2107 Group 8 Web Scraping Project

This notebook explains the warm-up first, then the Group 8 itch.io Top Sellers scraper. Code is kept in `warmup_books.py` and `scraper.py` so it can be run locally and automatically with GitHub Actions.

**Important:** replace the placeholder student email in the scripts before running. Check `robots.txt` first, wait 2–5 seconds between requests, and stop if the site blocks the scraper.

## Part A — Warm-up concepts

### Pagination
Pagination divides a long list into pages. The warm-up follows the listing page's `li.next a` link until no next page exists. This avoids stopping after the first 20 books.

### Detail pages
A listing card contains basic information and a link. The warm-up opens each book's detail page to collect extra fields from its product-information table. Pagination discovers items; detail-page scraping enriches each item.

In [ ]:
# Import the packages used in the warm-up and Group 8 project.
# requests downloads HTML; BeautifulSoup parses HTML; pandas helps inspect CSV data.
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

# Define the two sites used in the assignment.
BOOKS_START_URL = "https://books.toscrape.com/"
ITCHIO_START_URL = "https://itch.io/games/top-sellers"

print("Warm-up target:", BOOKS_START_URL)
print("Group 8 target:", ITCHIO_START_URL)

### How to run the warm-up

Run the following command in a terminal from this project folder:

```bash
python warmup_books.py
```

The script checks robots.txt, follows pagination, visits book detail pages, and writes `data/raw/books_warmup.csv`. The warm-up code is fully commented in `warmup_books.py`. It intentionally uses a delay between requests.

## Part B — Group 8 target reconnaissance

The assigned page is `https://itch.io/games/top-sellers`. The scraper uses the visible, sorted listing pages and assigns ranks by position in that ordered list. The current public `robots.txt` disallows paths such as `/search`, `/checkout/`, and download routes; the scraper checks permission for the top-sellers page before collecting data. Re-check robots.txt on the day you run it because site rules can change.

In [ ]:
# Inspect robots.txt before scraping. This cell only fetches the rules file.
# If it cannot be fetched, do not assume permission to scrape.
robots_url = "https://itch.io/robots.txt"
response = requests.get(robots_url, timeout=20, headers={"User-Agent": "DSC2107-Group8-Course-Project (your-email@example.com)"})
response.raise_for_status()
print(response.text)

In [ ]:
# Inspect one public listing page to understand its structure before running the full scraper.
# This is a one-page reconnaissance request, not a full collection.
headers = {"User-Agent": "DSC2107-Group8-Course-Project (your-email@example.com)"}
response = requests.get(ITCHIO_START_URL, headers=headers, timeout=20)
response.raise_for_status()
soup = BeautifulSoup(response.content, "html.parser")
print("Page title:", soup.title.get_text(strip=True) if soup.title else "No title found")
print("Possible game cards:", len(soup.select(".game_cell")))
print("Next page link:", [(a.get_text(" ", strip=True), a.get("href")) for a in soup.select("a.next_page, a[rel='next']")])

## Part C — Run the Group 8 scraper

The full scraper is in `scraper.py` and has comments explaining each function. It:

1. Checks robots.txt and uses a clear User-Agent.
2. Visits the top-sellers listing page and follows pagination.
3. Extracts rank, title, developer, price, genre, platforms, URL, and timestamp.
4. Waits 2–5 seconds between requests, retries transient errors with back-off, and stops on HTTP 403/429.
5. Saves raw CSV and JSON snapshots, a separate cleaned CSV, and an SQLite database with `products` and `observations` tables.

Run one collection with:

```bash
python scraper.py
```

A rank is the position in the sorted listing; it is not a published sales count. Missing fields stay missing instead of being guessed.

In [ ]:
# After running scraper.py, load the newest raw CSV to inspect the result.
# This does not make another web request.
raw_dir = Path("data/raw")
raw_files = sorted(raw_dir.glob("itchio_top_sellers_*.csv"))
if raw_files:
    latest_raw = raw_files[-1]
    sample = pd.read_csv(latest_raw)
    print("File:", latest_raw)
    print("Rows:", len(sample))
    display(sample.head(10))
else:
    print("No Group 8 raw CSV yet. Run `python scraper.py` first.")

## Part D — Daily automation

The file `.github/workflows/daily-scrape.yml` schedules a run every day at 06:30 UTC (09:30 in Uganda). It commits daily outputs so observations persist between hosted runners. To enable it, upload this folder to a private GitHub repository, set the Actions variable `STUDENT_EMAIL`, allow workflow write permissions, enable Actions, and run one manual test. Monitor every run. Disable the workflow after seven consecutive daily runs for the assignment.

## Part E — Suggested analysis questions

Once seven daily runs exist, use the database or cleaned CSV files to answer:

- What is the distribution of listed prices? Treat free/missing values carefully.
- Which genres and platforms appear most often in the top 500?
- Which games move most in rank between days?
- How often do games disappear from or enter the top 500?
- How stable are developer representation and price across the collection period?

Create at least six labelled charts in the final analysis notebook, explain what each chart shows, and discuss limitations. A top-seller rank is an ordinal ranking and should not be interpreted as exact sales volume.